In [ ]:
import numpy as np 
import pandas as pd 
import datetime
import os 
import pickle
import sys 
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt 
# %matplotlib notebook
sys.path.insert(1,'../../../../arimaclust/')
from arimaclust.models.centroid import ConsistentARMAClustering
from arimaclust.models.neighbours import NearestNeighbourModel
from arimaclust.autoarima.estimation import estimate_processes
from arimaclust.metrics.matrix import dist_matrix_parallel
from arimaclust.tracking.mapping import mapp_hungarian
from arimaclust.utils import get_equal_time_periods



In [ ]:
df_usd = pd.read_csv('data/final_usd_2010_2020.csv',index_col='Unnamed: 0')

In [ ]:
df_gdp = pd.read_html('https://ceoworld.biz/2022/03/31/economy-rankings-largest-countries-by-gdp-2022/')[0]
df_gdp['Country'] = df_gdp['Country'].apply(lambda x: x.lower())
# df_codes = pd.read_html('https://www.iban.com/currency-codes')[0]
# df_codes['Country'] = df_codes['Country'].apply(lambda x: x.lower())
df_codes = pd.read_csv('codes.csv')
df_merged = pd.merge(left=df_codes,right=df_gdp,on='Country').sort_values('Rank')
top_codes = df_merged[df_merged['Rank']<=100]['Code'].unique()
top_codes = [tp for tp in top_codes if tp not in ['MXV', 'CHE', 'CHW', 'COU', 'UYI', 'VED', 'BOV']]

In [ ]:
df_usd = df_usd[top_codes]

## Drop some symbols 

In [ ]:
# drop symbols with unknown behaviour (picewise constants)
to_drop = ['ZMK','BYR','LVL','TMT','CLF','ZWL','JEP','CUP','ERN','CUC','USD','BMD','BBD','BSD','SDG','KPW','PAB']
df_usd.drop(columns=to_drop,inplace=True,errors='ignore')

# symbols with too much nans 
to_much_nans = ['BYN','IMP','GGP','ZMW','KYD']
df_usd.drop(columns=to_much_nans,inplace=True,errors='ignore')

# metals 
metals = ['XAU','XAG']
df_usd.drop(columns=metals,inplace=True,errors='ignore')

# coins 
coins = ['BTC']
df_usd.drop(columns=coins,inplace=True,errors='ignore')


## We have nans in some series 

In [ ]:
nans = df_usd.isna().sum().sort_values(ascending=False)
nan_symb = nans.index[nans!=0]

In [ ]:
for ns in nan_symb:
    df_usd[ns].plot()
    
    plt.title(f'{ns}:{nans[ns]}')
    plt.show()

In [ ]:
df_usd[df_usd['IDR'].isna()]

## Fill with interpolation

In [ ]:
for ns in nan_symb:
    
    l,u = df_usd[df_usd[ns].isna()].index.min(),df_usd[df_usd[ns].isna()].index.max()
    inter = df_usd[ns].interpolate()  
    df_usd[l:u][ns].plot(style='r-',label='True')
    inter[l:u].plot(style='g--',label='Interpolated')
    plt.title(f'{ns}:{nans[ns]}')
    plt.legend()
    plt.show()

In [ ]:
df_usd = df_usd.interpolate()

In [ ]:
df_usd

## Some utils

In [ ]:
from pmdarima.preprocessing import BoxCoxEndogTransformer


def prepare_data(df,bc=True,ws=None):
    X = []
    for col in df.columns:
        if ws:
            x = df[col].rolling(ws).mean().dropna().values
        else:
            x = df[col].dropna().values
        if bc:
            print(col)
            x, _ = BoxCoxEndogTransformer(lmbda2=1e-6).fit_transform(x)

        X.append(x)
    return X

def plot_mds(dist_matrix,labels,names,title=None,medoids=None):
    from sklearn import manifold 

    mds = manifold.MDS(
        n_components=2,
        max_iter=3000,
        eps=1e-9,
#         random_state=42,
        dissimilarity="precomputed",
        n_jobs=-1,
        normalized_stress="auto",
    )
    pos = mds.fit(dist_matrix).embedding_


    nmds = manifold.MDS(
        n_components=2,
        metric=False,
        max_iter=3000,
        eps=1e-12,
        dissimilarity="precomputed",
#         random_state=42,
        n_jobs=-1,
        n_init=1,
        normalized_stress="auto",
    )
    pos = nmds.fit_transform(dist_matrix, init=pos)
    
    fig, ax = plt.subplots(figsize=(12,8))
    
    
    
    cmap = plt.get_cmap('viridis', np.max(labels) + 1)

    ax.scatter(pos[:,0],pos[:,1], c=labels,cmap=cmap, marker='o')
    
    for i in range(len(names)):
        ax.annotate(names[i], (pos[i][0], pos[i][1]),fontsize=8)
    if medoids:
        markers_on = [i for i in range(len(names)) if names[i] in medoids]
        ax.scatter(pos[markers_on,0], pos[markers_on, 1], marker='s', s=100, facecolors='none', edgecolors='black')
    if title:
        ax.set_title(title)

        
def plot_clusters(df,labels,k):
    for i in range(k):
        df[df.columns[labels==i]].plot()
        plt.legend(bbox_to_anchor=(0, 1, 1, 0), ncol=2)
        plt.show()
        
def load_pickle(path):
    f = open(path,'rb')
    return pickle.load(f)


def get_neibours(df,dist_matrices,cols,symb,delta=6,r=0.3):
    start, end = datetime.datetime(year=2010, day=1, month=1), datetime.datetime(year=2020, day=1, month=1)
    periods = get_equal_time_periods(start_date=start, end_date=end, num_periods=delta)
     
    for i,(current_start, current_end) in enumerate(periods):
        
        dm_at_i = dist_matrices[i]
        
        df_part = df[str(current_start):str(current_end)].copy()
        
        cols_at_i = cols[i]
        symb_ind = cols_at_i.index(symb)
        
        nn = NearestNeighbors(radius=r,metric='precomputed')
        nn.fit(dm_at_i)
        distances, indices = nn.radius_neighbors()
        
        neighbors = np.array(cols_at_i)[indices[symb_ind]].tolist() + [symb]
        dists = distances[symb_ind].tolist() + [0]
        
        try:
            print(current_start,current_end)
#             df_part.pct_change()[neighbors].rolling(2).mean().plot()
            df_part[neighbors].plot()
            plt.legend([f'{n}: {np.round(d,3)}' for n,d in zip(neighbors,dists)])
            plt.show()
        except Exception as e:
            continue
        current_start = current_end
        
def find_medoid(dist_matrix, labels, cluster_ind,columns):
    cluster_points = np.where(labels == cluster_ind)[0]
    distances = dist_matrix[cluster_points][:, cluster_points]
    medoid_idx = np.argmin(np.sum(distances, axis=1))
    medoid = cluster_points[medoid_idx]
    
    return columns[medoid]



def plot_cluster_changes(labels,cols):

    fig, ax = plt.subplots(figsize=(12,8))


    step = 15
    coords = []
    for j in range(len(labels)):

        labels_at_j = labels[j]
        
        cols_at_j = np.array(cols[j])
        coords_at_j = {}
        min_range = 0

        for i in range(int(labels_at_j.max())+1):
            
            cluster = cols_at_j[labels_at_j==i]

            x_ = np.arange(min_range,min_range+len(cluster)*step,step)

            pts = np.array([[x_coord,j*2] for x_coord in x_])

            for c,coord in zip(cluster,pts):
                coords_at_j[c] = coord

            ax.scatter(pts[:,0],pts[:,1])

            min_range = min_range+int(len(cluster)*step)+2
        coords.append(coords_at_j)

    for j in range(len(labels))[::-1]:
        if j==0:
            break
        labels_at_j = labels[j]
        cols_at_j = np.array(cols[j])
        n_c = int(labels_at_j.max())
        coords_at_j = coords[j]
        coords_at_previous = coords[j-1]
        for i in range(n_c):
            cluster = cols_at_j[labels_at_j==i]
            color = next(ax._get_lines.prop_cycler)['color']
            for j,symb in enumerate(cluster):
                if symb not in coords_at_previous:
                    continue
                else:
                    x_ = [coords_at_j[symb][0],coords_at_previous[symb][0]]
                    y_ = [coords_at_j[symb][1],coords_at_previous[symb][1]]
                ax.plot(x_,y_,color=color)


def match_overlap(labels1,labels2,cols1,cols2):
    n_c1 = max(labels1)+1
    n_c2 = max(labels2)+1
    m = np.zeros((n_c1,n_c2))
    for i in range(n_c1):
        for j in range(n_c2):
            cluster1 = np.array(cols1)[labels1==i]
            cluster2 = np.array(cols2)[labels2==j]
            intersection = set(cluster1).intersection(set(cluster2))
            m[i,j] = len(intersection)
    print(m)
    r,c = linear_sum_assignment(-m)
    print(r,c)
    return r,c


def track_cluster(initial_ind,labels,cols):
    ind = initial_ind
    cluster = [np.array(cols[0])[labels[0]==initial_ind].tolist()]
    for i in range(1,len(labels)):
        r,c = match_overlap(labels1=labels[i-1],labels2=labels[i],cols1=cols[i-1],cols2=cols[i])
        ind = c[ind]
        cluster.append(np.array(cols[i])[labels[i]==ind].tolist())
        
#         ind = r[ind]
    return cluster
        
    
from sklearn.metrics import pairwise_distances
import numpy as np
from scipy.spatial.distance import pdist, squareform

def dunn_index(dist_matrix, labels):
    """
    Calculate the Dunn index for a given clustering solution.

    Parameters:
    -----------
    X : array-like, shape (n_samples, n_features)
        The input data.

    labels : array-like, shape (n_samples,)
        The labels of the clustering solution.

    Returns:
    --------
    float
        The Dunn index value.
    """
    # Calculate the distance matrix between samples

    # Calculate the minimum distance between points in different clusters
    inter_cluster_dists = []
    for i in np.unique(labels):
        indices = np.where(labels == i)[0]
        other_indices = np.where(labels != i)[0]
        inter_cluster_dists.append(np.min(dist_matrix[indices][:, other_indices]))

    # Calculate the maximum diameter of each cluster
    intra_cluster_diameters = []
    for i in np.unique(labels):
        indices = np.where(labels == i)[0]
        cluster_dists = dist_matrix[indices][:, indices]
        intra_cluster_diameters.append(np.max(cluster_dists))

    # Calculate the Dunn index
    dunn_index = np.min(inter_cluster_dists) / np.max(intra_cluster_diameters)

    return dunn_index


def read_results(exp_path):
    all_files = os.listdir(exp_path)

    labels_paths = sorted([os.path.join(exp_path,path) for path in all_files if 'label' in path])
    matrices_paths = sorted([os.path.join(exp_path,path) for path in all_files if 'matrix' in path])
    cols_paths = sorted([os.path.join(exp_path,path) for path in all_files if 'cols' in path])

    cols = [load_pickle(col_path) for col_path in cols_paths]
    labels = [np.load(path) for path in labels_paths]
    dist_matrices = [np.load(path) for path in matrices_paths]
    return cols,labels,dist_matrices

## Viz results


In [ ]:
import pickle

exp_path = 'results2/'
all_files = os.listdir(exp_path)

labels_paths = sorted([os.path.join(exp_path,path) for path in all_files if 'label' in path])
matrices_paths = sorted([os.path.join(exp_path,path) for path in all_files if 'matrix' in path])
cols_paths = sorted([os.path.join(exp_path,path) for path in all_files if 'cols' in path])

cols = [load_pickle(col_path) for col_path in cols_paths]
labels = [np.load(path) for path in labels_paths]
dist_matrices = [np.load(path) for path in matrices_paths]


In [ ]:
labels_paths,cols_paths

In [ ]:
plot_cluster_changes(labels=labels,cols=cols)

In [ ]:
track_cluster(initial_ind=6,labels=labels,cols=cols)

# Viz 3

In [ ]:
import pickle

exp_path = 'results4/'


In [ ]:
cols,labels,dist_matrices = read_results(exp_path)

In [ ]:
t_ind = 1
medoids = []
for c_ind in range(5):
    med = find_medoid(dist_matrix=dist_matrices[t_ind],labels=labels[t_ind],cluster_ind=c_ind,columns=cols[t_ind])
    medoids.append(med)
    print(f'c:{c_ind},t:{t_ind}, med: {med}')

In [ ]:
get_neibours(df = df_usd,labels=labels,dist_matrices=dist_matrices,cols=cols,symb='CHF',r=0.2)

In [ ]:
df_usd['2018-01-01':'2019-01-01'][['CZK','CHF']].plot()

In [ ]:
t_ind = 9

medoids = []
for c_ind in range(5):
    med = find_medoid(dist_matrix=dist_matrices[t_ind],labels=labels[t_ind],cluster_ind=c_ind,columns=cols[t_ind])
    medoids.append(med)
    print(f'c:{c_ind},t:{t_ind}, med: {med}')
    
plot_mds(labels=labels[t_ind],dist_matrix=dist_matrices[t_ind],names=cols[t_ind],medoids=medoids)

# Estimate kappa

In [ ]:
dunn_index(dist_matrices[0],labels[0])

In [ ]:
from arimaclust.models.centroid import ConsistentARMAClustering

In [ ]:
labels_dunn = []
medoids_dunn = []
k_range = [2,3,4,5,6,7,8,9,10]

for t_ind in range(10):
    dunn_indices_per_t = []
    medoids_per_t = []
    
    for k in k_range:
        model = ConsistentARMAClustering(k=k,metric='precomputed')
        model.fit(dist_matrices[t_ind])
        dunn_indices_per_t.append(dunn_index(dist_matrices[t_ind],model.labels_))
    

    
    max_ = np.max(dunn_indices_per_t)
    k_opt = k_range[np.where(dunn_indices_per_t == max_)[0].max()]
    
    plt.plot(k_range,dunn_indices_per_t)
    plt.title(f'{t_ind}:{k_opt}')
    plt.show()
    
    model_t = ConsistentARMAClustering(k=k_opt,metric='precomputed')
    model_t.fit(dist_matrices[t_ind])
    labels_dunn.append(model_t.labels_)
    for c_ind in range(k_opt):
        med = find_medoid(dist_matrix=dist_matrices[t_ind],labels=model_t.labels_,cluster_ind=c_ind,columns=cols[t_ind])
        medoids_per_t.append(med)
    medoids_dunn.append(medoids_per_t)

In [ ]:
labels_dunn[0]

In [ ]:
for t_ind in range(10):
    plot_mds(labels=labels_dunn[t_ind],dist_matrix=dist_matrices[t_ind],names=cols[t_ind],medoids=medoids_dunn[t_ind],title=f'{t_ind}:{labels_dunn[t_ind].max()+1}')
    plt.show()

In [ ]:
plot_cluster_changes(labels=labels_dunn,cols=cols)

In [ ]:
track_cluster(initial_ind=0,labels=labels_dunn,cols=cols)

In [ ]:
get_neibours(df = df_usd,labels=labels,dist_matrices=dist_matrices,cols=cols,symb='CHF',r=0.1)

In [ ]:
import pickle

exp_path = 'results_out_sample/'
all_files = os.listdir(exp_path)

labels_paths = sorted([os.path.join(exp_path,path) for path in all_files if 'label' in path])
matrices_paths = sorted([os.path.join(exp_path,path) for path in all_files if 'matrix' in path])
cols_paths = sorted([os.path.join(exp_path,path) for path in all_files if 'cols' in path])

cols = [load_pickle(col_path) for col_path in cols_paths]
labels = [np.load(path) for path in labels_paths]
dist_matrices = [np.load(path) for path in matrices_paths]


In [ ]:
labels_dunn = []
medoids_dunn = []
k_range = [2,3,4,5,6,7,8,9,10]

for t_ind in range(10):
    dunn_indices_per_t = []
    medoids_per_t = []
    
    for k in k_range:
        model = ConsistentARMAClustering(k=k,metric='precomputed')
        model.fit(dist_matrices[t_ind])
        dunn_indices_per_t.append(dunn_index(dist_matrices[t_ind],model.labels_))
    

    
    max_ = np.max(dunn_indices_per_t)
    k_opt = k_range[np.where(dunn_indices_per_t == max_)[0].max()]
    
    plt.plot(k_range,dunn_indices_per_t)
    plt.title(f'{t_ind}:{k_opt}')
    plt.show()
    
    model_t = ConsistentARMAClustering(k=k_opt,metric='precomputed')
    model_t.fit(dist_matrices[t_ind])
    labels_dunn.append(model_t.labels_)
    for c_ind in range(k_opt):
        med = find_medoid(dist_matrix=dist_matrices[t_ind],labels=model_t.labels_,cluster_ind=c_ind,columns=cols[t_ind])
        medoids_per_t.append(med)
    medoids_dunn.append(medoids_per_t)

In [ ]:
get_neibours(df = df_usd,labels=labels,dist_matrices=dist_matrices,cols=cols,symb='CHF',r=0.1)

In [ ]:
from arimaclust.autoarima.estimation import estimate_processes, fit_arma
from arimaclust.metrics.matrix import dist_matrix_parallel
import pmdarima as pm

In [ ]:
def prepare_data(df, bc=False, ws=None):
    X = []
    for col in df.columns:
        if ws:
            x = df[col].rolling(ws).mean().dropna().values
        else:
            x = df[col].dropna().values
        if bc:
            x, _ = BoxCoxEndogTransformer(lmbda2=1e-6).fit_transform(x)
        X.append(x)
    return X



process_args = {'max_p': 3,
                'max_q': 3,
                'trace': True,
                'd': 0
                }

metric_args = {'tau': 30}


df_part = df_usd['2015-01-01':'2016-01-01'].copy()

    # drop zero columns
    # df_part.drop(columns=[col for col in df_part.columns if df_part[col].nunique() == 1], inplace=True)

cols = df_part.columns.tolist()
# df_part = np.log1p(df_part.pct_change())

X = prepare_data(df_part,ws=2)

processes = estimate_processes(X,**process_args)
dm = dist_matrix_parallel(processes=processes,metric_name='piccolo', **metric_args)

In [ ]:
get_neibours(df=df_usd,dist_matrices=[dm],cols=[list(df_part.columns)],symb='CHF',r=0.5)

In [ ]:
df_part = df_usd['2015-01-01':'2015-06-01'][['CHF','NZD']].copy()
# df_part = np.log1p(df_part.pct_change())


In [ ]:
df_part

In [ ]:

process_args = {'max_p': 3,
                'max_q': 3,
                'trace': True,
                'd': 0
                }


x_pm_model = pm.auto_arima(df_part['CHF'].dropna().values, error_action='ignore',
                               stepwise=True,
                               suppress_warnings=True,
                               
                               **process_args
                               )

In [ ]:
x_pm_model.summary()

In [ ]:
for t_ind in range(10):
    plot_mds(labels=labels_dunn[t_ind],dist_matrix=dist_matrices[t_ind],names=cols[t_ind],medoids=medoids_dunn[t_ind],title=f'{t_ind}:{labels_dunn[t_ind].max()+1}')
    plt.show()

# witout log tr

In [ ]:
exp_path='result_without_log/'
cols,labels,dist_matrices = read_results(exp_path)

In [ ]:
labels_dunn = []
medoids_dunn = []
k_range = [3,4,5,6,7]

fig, ax = plt.subplots(figsize=(12,8))

for t_ind in range(10):
    dunn_indices_per_t = []
    medoids_per_t = []
    
    for k in k_range:
        model = ConsistentARMAClustering(k=k,metric='precomputed')
        model.fit(dist_matrices[t_ind])
        dunn_indices_per_t.append(dunn_index(dist_matrices[t_ind],model.labels_))
    

    
    max_ = np.max(dunn_indices_per_t)
    k_opt_ind = np.where(dunn_indices_per_t == max_)[0].max()
    k_opt = k_range[k_opt_ind]
    
    ax.plot(k_range,dunn_indices_per_t,label=t_ind)
    ax.scatter(k_opt,dunn_indices_per_t[k_range.index(k_opt)])
    
    model_t = ConsistentARMAClustering(k=k_opt,metric='precomputed')
    model_t.fit(dist_matrices[t_ind])
    labels_dunn.append(model_t.labels_)
    for c_ind in range(k_opt):
        med = find_medoid(dist_matrix=dist_matrices[t_ind],labels=model_t.labels_,cluster_ind=c_ind,columns=cols[t_ind])
        medoids_per_t.append(med)
    medoids_dunn.append(medoids_per_t)

plt.legend()
plt.show()

In [ ]:
for t_ind in range(10):
    plot_mds(labels=labels_dunn[t_ind],dist_matrix=dist_matrices[t_ind],names=cols[t_ind],medoids=medoids_dunn[t_ind],title=f'{t_ind}:{labels_dunn[t_ind].max()+1}')
    plt.show()

In [ ]:
labels_dunn[0][list(df_usd.columns).index('PEN')]

In [ ]:
np.array(cols[0])[labels_dunn[0] == 1]

# 6 months

In [ ]:
exp_path = 'results_6_months'
cols,labels,dist_matrices = read_results(exp_path)

In [ ]:
from sklearn.metrics import silhouette_score

labels_dunn = []
medoids_dunn = []
k_range = [3,4,5,6,7,8,9,10]

fig, ax = plt.subplots(figsize=(12,8))

for t_ind in range(len(labels)):
    dunn_indices_per_t = []
    medoids_per_t = []
    
    for k in k_range:
        model = ConsistentARMAClustering(k=k,metric='precomputed')
        model.fit(dist_matrices[t_ind])
        score = silhouette_score(dist_matrices[t_ind],model.labels_, metric='precomputed')
#         score = dunn_index(dist_matrices[t_ind],model.labels_)
        dunn_indices_per_t.append(score)
    

    
    max_ = np.max(dunn_indices_per_t)
    k_opt_ind = np.where(dunn_indices_per_t == max_)[0].max()
    k_opt = k_range[k_opt_ind]
    
    ax.plot(k_range,dunn_indices_per_t,label=t_ind)
    ax.scatter(k_opt,dunn_indices_per_t[k_range.index(k_opt)])
    
    model_t = ConsistentARMAClustering(k=k_opt,metric='precomputed',metric_args={"tau":100})
    model_t.fit(dist_matrices[t_ind])
    labels_dunn.append(model_t.labels_)
    for c_ind in range(k_opt):
        med = find_medoid(dist_matrix=dist_matrices[t_ind],labels=model_t.labels_,cluster_ind=c_ind,columns=cols[t_ind])
        medoids_per_t.append(med)
    medoids_dunn.append(medoids_per_t)

plt.legend()
plt.show()

In [ ]:
for t_ind in range(len(labels)):
    plot_mds(labels=labels_dunn[t_ind],dist_matrix=dist_matrices[t_ind],names=cols[t_ind],medoids=medoids_dunn[t_ind],title=f'{t_ind}:{labels_dunn[t_ind].max()+1}')
    plt.show()

In [ ]:
plot_cluster_changes(labels=labels_dunn[:5],cols=cols[:5])

In [ ]:
import numpy as np

def intra_cluster_distance(dist_matrix, labels):
    """
    Calculates the average intra-cluster distance for each cluster.
    Parameters:
        dist_matrix (numpy.ndarray): Distance matrix of the data points
        labels (numpy.ndarray): Cluster labels for each data point
    Returns:
        numpy.ndarray: Array containing the average intra-cluster distance for each cluster
    """
    unique_labels = np.unique(labels)
    n_clusters = len(unique_labels)
    intra_cluster_distances = np.zeros(n_clusters)
    for i, label in enumerate(unique_labels):
        cluster_indices = np.where(labels == label)[0]
        cluster_size = len(cluster_indices)
        if cluster_size > 1:
            # Calculate the pairwise distances between the points in the cluster
            distances = dist_matrix[np.ix_(cluster_indices, cluster_indices)]
            # Calculate the average distance between the points in the cluster
            intra_cluster_distances[i] = np.sum(distances) / (cluster_size * (cluster_size - 1))
        else:
            # If the cluster contains only one point, set the intra-cluster distance to 0
            intra_cluster_distances[i] = 0
    return intra_cluster_distances


In [ ]:
for i in range(len(labels_dunn)):
    print(intra_cluster_distance(dist_matrices[i],labels_dunn[i]))

In [ ]:
np.array(cols[0])[labels_dunn[0]==3]

In [ ]:
np.array(cols[2])[labels_dunn[2]==3]

In [ ]:


def preprocess_data(df):
    # drop symbols with unknown behaviour (piecewise constants)
    to_drop = ['ZMK', 'BYR', 'LVL', 'TMT', 'CLF', 'ZWL', 'JEP', 'CUP', 'ERN', 'CUC', 'USD', 'BMD', 'BBD', 'BSD', 'SDG',
               'KPW', 'PAB']
    df.drop(columns=to_drop, inplace=True, errors='ignore')

    # symbols with to mooch nans
    to_much_nans = ['BYN', 'IMP', 'GGP', 'ZMW', 'KYD']
    df.drop(columns=to_much_nans, inplace=True, errors='ignore')

    # metals
    metals = ['XAU', 'XAG']
    df.drop(columns=metals, inplace=True, errors='ignore')

    # coins
    coins = ['BTC']
    df.drop(columns=coins, inplace=True, errors='ignore')
    df = df.interpolate()
    return df


def prepare_data(df, bc=False, ws=None):
    X = []
    for col in df.columns:
        if ws:
            x = df[col].rolling(ws).mean().dropna().values
        else:
            x = df[col].dropna().values
        if bc:
            x, _ = BoxCoxEndogTransformer(lmbda2=1e-6).fit_transform(x)
        X.append(x)
    return X


df_path = 'data/final_usd_2010_2020.csv'

df_gdp = pd.read_html('https://ceoworld.biz/2022/03/31/economy-rankings-largest-countries-by-gdp-2022/')[0]
df_gdp['Country'] = df_gdp['Country'].apply(lambda x: x.lower())
# df_codes = pd.read_html('https://www.iban.com/currency-codes')[0]
# df_codes['Country'] = df_codes['Country'].apply(lambda x: x.lower())
df_codes = pd.read_csv('codes.csv')
df_merged = pd.merge(left=df_codes, right=df_gdp, on='Country').sort_values('Rank')
top_codes = df_merged[df_merged['Rank'] <= 100]['Code'].unique()
top_codes = [tp for tp in top_codes if tp not in ['MXV', 'CHE', 'CHW', 'COU', 'UYI', 'VED', 'BOV']]
df_usd = pd.read_csv(df_path, index_col='Unnamed: 0')

df_usd = df_usd[top_codes]
df_usd = preprocess_data(df_usd)

process_args = {'max_p': 3,
                'max_q': 3,
                'trace': True,
                'd': 0
                }

metric_args = {'tau': 30}


df_part = df_usd['2010-01-01':'2010-07-01'].copy()

    # drop zero columns
df_part.drop(columns=[col for col in df_part.columns if df_part[col].nunique() == 1], inplace=True)

cols_part = df_part.columns.tolist()
df_part = np.log1p(df_part.pct_change())

X = prepare_data(df_part,ws=None)

processes = estimate_processes(X,**process_args)
dm = dist_matrix_parallel(processes=processes,metric_name='piccolo', **metric_args)

In [ ]:
dm

In [ ]:
processes[cols_part.index("CLP")].arma2ar(15)

In [ ]:
processes[cols_part.index("AED")].arma2ar(15)

In [ ]:
processes[cols_part.index("TND")].arma2ar(40)